# ___Correlated evolution between a continuous trait and a discrete categorical trait___
--------------------------------------

#### ___References___
----------------

- ___[`OUwie::hOUwieStarterGuide`](https://thej022214.github.io/OUwie/articles/hOUwieStarterGuide.html)___     
- ___[`OUwie`](https://thej022214.github.io/OUwie/index.html)___     
- ___[A novel method for jointly modeling the evolution of discrete and continuous traits, Evolution, 77(3), pp. 836–851](https://doi.org/10.1093/evolut/qpad002)___    

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [2]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("nlme")
    library("corHMM")
    library("geiger")
    # library("mkcor")
    library("OUwie")
    library("reshape2")
    library("ggplot2")
    library("U.PhyloMaker")
})

stopifnot(packageVersion("OUwie") == "2.16")
stopifnot(packageVersion("corHMM") == "2.8")

## ___Model fitting to dummy data___
-----------------

In [16]:
# example model fitting

data(tworegime)
dat <- data.frame(sp = tree$tip.label, X = sample(c(0, 1, 2), length(tree$tip.label), replace = TRUE), Y = sample(c(0, 1), length(tree$tip.label), replace = TRUE), FS = rnorm(length(tree$tip.label), 10, 3))
head(dat)

,sp,X,Y,FS
,<chr>,<dbl>,<dbl>,<dbl>
1,t1,0,1,9.365061
2,t2,0,0,10.502899
3,t3,2,0,5.821208
4,t4,0,0,9.402193
5,t5,1,1,11.637713
6,t6,0,0,8.751180


In [17]:
p <- c(0.01670113, 0.39489947, 0.18619839, 1.67259459, 0.16817414)  # my fixed set of parameters
pp_oum <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25, p = p)  # you likely won't use this p argument

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Calculating likelihood from a set of fixed parameters.
[1]  0.01670113  0.39489947  0.18619839 51.67259459 50.16817414


In [17]:
pp_oum


Fit
    lnLTot   lnLDisc   lnLCont     AIC     AICc      BIC nTaxa nPars
 -25.15545 -5.260483 -18.62633 60.3109 61.34539 71.10532    64     5

Legend
  1   2 
"1" "2" 

Regime Rate matrix
           (1)        (2)
(1)         NA 0.01670113
(2) 0.01670113         NA

OU Estimates
             (1)       (2)
alpha  0.3948995 0.3948995
sigma2 0.1861984 0.1861984
theta  1.6725946 0.1681741


Half-life (another way of reporting alpha)
    (1)     (2) 
1.75525 1.75525 

In [6]:
# fitting without p
model <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) 

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


## ___Fitting the `hOUwie` model to our data___
-------------------------

In [50]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal")
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

In [2]:
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = '')) # MERGED contains '/'
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))

data <- data[matched_row_indices, ]
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [3]:
# OUwie::hOUwie expects the columns in the following order => species name, categorical trait, continuous trait

RDdata <- data[, c("binominal", "myco", "RD")] # for root diameter

In [7]:
unique(RDdata$myco) # 6 unique values for mycorrhizal states

[1] "AMNM"  "AM"    "ErM"   "NM"    "AMEcM" "EcM"

In [10]:
lapply(X = split(x = RDdata, f = ~myco), FUN = function (df) mean(df$RD)) |> as.data.frame()

AM,AMEcM,AMNM,EcM,ErM,NM
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0.3465879,0.2019064,1.791249,0.2483341,0.09060939,0.2205958


In [ ]:
#--------
# NOTES
#--------

# - all our hypotheses are based on correlations between continuous characters and discrete characters
# - we do not care about differences in rates of trait evolution, or differences in the regime dependent pull towards the continuous trait optima or do we???
# - so the type of OU model we need is the one that allows for variations in the continuous trait optimum based on on the discrete regime (trait) => choose the "OUM" model
# - the paper also shows an example of fleshy/dry fruit types (discrete categorical trait) being phylogenetically correlated to the humidity levels (continuous trait) of the species origin
# - since we hypothesize that these two traits are correlated, then our hypothesis calls for character dependence (CD)
# - i.e CD models allow the parameters that define the continuous trait evolution to vary based on the discrete character regime
# - we could then test our null hypothesis by fitting a character independent (CID) model, where the OU parameters are same across the phylogeny, irrespective of the discrete trait regime
# - or alternatively, we could fit a CID+ model, that allows variation in the OU parameters BUT not predicated on the discrete character regime, this variations come from unobserved hidden states

# model of choice for continuous character evolution => OUM
# we do not want the BM family of OU models because they only allow for differences in rate of stochastic evolution
# type of models to fit => CD, CID & CID+
# 

In [ ]:
#---------
# NOTES
#--------

# look up https://thej022214.github.io/OUwie/reference/hOUwie.html before specifying the discrete_model, continuous_model arguments to OUwie::hOUwie()
# code for paper Boyko, J.D., O’Meara, B.C. and Beaulieu, J.M. (2023) “A novel method for jointly modeling the evolution of discrete and continuous traits,” Evolution, 77(3), pp. 836–851. can be found at
# https://github.com/jboyko/2020_houwie/tree/master

# discrete_model - Either a user-supplied index of parameters to be optimized or one of "ARD", "SYM", or "ER". 
# ARD: all rates differ. SYM: rates between any two states do not differ. ER: all rates are equal.
# continuous_model - Either a user-supplied index matrix specifying the continuous model parameters to be estimated or one of "BM1", "BMV", "OU1", "OUA", "OUV", "OUM", "OUVA", "OUMV", "OUMA", "OUMVA" 
# (See also getOUParamStructure).
# null.model - A boolean indicating whether the model being run is a character-independent model with rate heterogeneity. Rate.cat must be greater than 1.
# nSim - The number of stochastic maps evaluated per iteration of the ML search.

In [11]:
OUwie::getOUParamStructure("OUM", nObsState = 6) # for OUM models alpha and sigma square are kept constant and only the theta (continuous trait optimum) is allowed to vary

,(1),(2),(3),(4),(5),(6)
alpha,1,1,1,1,1,1
sigma2,2,2,2,2,2,2
theta,3,4,5,6,7,8


In [13]:
OUwie::getOUParamStructure("OUMA", nObsState = 6) # alpha and theta varies, sigma square is kept constant

,(1),(2),(3),(4),(5),(6)
alpha,1,2,3,4,5,6
sigma2,7,7,7,7,7,7
theta,8,9,10,11,12,13


In [15]:
OUwie::getOUParamStructure("OUMVA", nObsState = 6) # alpha, sigma square & theta are allowed to vary

,(1),(2),(3),(4),(5),(6)
alpha,1,2,3,4,5,6
sigma2,7,8,9,10,11,12
theta,13,14,15,16,17,18


In [20]:
OUwie::getOUParamStructure("BM1", nObsState = 6) # rate change is not allowed in BM models
# the other two params (sigma square and theta) are kept constant

,(1),(2),(3),(4),(5),(6)
alpha,NA,NA,NA,NA,NA,NA
sigma2,1,1,1,1,1,1
theta,2,2,2,2,2,2


In [22]:
OUwie::getOUParamStructure("BMV", nObsState = 6) # rate change is not allowed in BM models
# sigma square is allowed to vary BUT theta is NOT

,(1),(2),(3),(4),(5),(6)
alpha,NA,NA,NA,NA,NA,NA
sigma2,1,2,3,4,5,6
theta,7,7,7,7,7,7


### ___Correlated evolotion of root diameter & mycorrhizal states___
------------------------

In [ ]:
#-------
# NOTES
#-------

# from https://thej022214.github.io/OUwie/reference/hOUwie.html

# rate.cat =>
# default is 1 rate class in which only observed discrete states evolve.
# however, there are two main reasons one may be interested in increasing the number of rate categories. 
# first, rate heterogeneity throughout the tree of life is more of a rule than a possibility. not all things will evolve in the same way at the same time.
# for example, woody and herbaceous plants may evolve one way on the mainland and a completely different way on island.
# the challenge then is to allow for heterogeneity in the evolutionary process when we do not know what the 'mainland' or 'island' variables are, this is what hidden Markov models allow us to do 
# the second reason to include hidden rate categories is to reduce the error rates of finding a false correlation, this has been discussed elsewhere in depth (see Maddison and FitzJohn 2015, Uyeda et al. 2018, Boyko and Beaulieu 2022). 
# the problem lies in if we compare models with rate heterogeneity to models without it, most character dependent models (correlation models) allow for different ways for the characters to evolve.
# in fact, their reliable inference depends on being able to define the differences between, for example, body size evolution on islands and mainlands. 
# however, most character-independent models have no way to allow for variable ways for body size to evolve, so, whether or not the evolution of body size is connected to island systems, 
# we may be biased towards selecting correlation models simply because they allow body size to have different rates of evolution.

In [ ]:
#----------
# NOTES
#---------

# CD models => evolution of discrete character states and the evolution of continuous trait is correlated. i.e allow MVA to vary depending on discrete state transitions
# HYB models => (CID+) allow certain parameters of continuous trait evolution to vary depending on the discrete character state transitions
# CID models => CID models are models that assume no direct correlation between the evolution of discrete character states and the continuous trait
# IF null.model WAS SET TO TRUE, rate.cat MUST BE GREATER THAN 1 TO ALLOW HIDDEN MARKOV PROCESSES

In [37]:
# with discrete character dependence (CD) AND with hidden character variations => HYB (hybrid) models, accomplished by null.model = TRUE & rate.cat > 1
# basic CD models => null.model = TRUE & rate.cat = 1
# "From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models"????? REPORT THE VERSION OF THE PACKAGE IN WRITING
# THIS ERROR COMES FROM CORHMM NOT OUWIE!!!

tm <- Sys.time()

# rate.cat = 1 and null.model = FALSE 

ER_OUM_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ER_OUMA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

SYM_OUM_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE)
SYM_OUMA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_RD_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = FALSE
# WHAT'S THE POINT OF THESE MODELS??? WHAT'S THE POINT OF ALLOWING 2 RATE CATEGORIES WITHOUT ALLOWING A CID PROCESS OF EVOLUTION????

# ER_OUM_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE)
# ER_OUMA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
# ER_OUMV_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
# ER_OUMVA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)
# 
# ARD_OUM_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
# ARD_OUMA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
# ARD_OUMV_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
# ARD_OUMVA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)
# 
# SYM_OUM_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE)
# SYM_OUMA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
# SYM_OUMV_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
# SYM_OUMVA_RD <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = TRUE

ER_OUM_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ER_OUMA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ER_OUMV_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ER_OUMVA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

ARD_OUM_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ARD_OUMA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ARD_OUMV_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ARD_OUMVA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

SYM_OUM_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = TRUE)
SYM_OUMA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
SYM_OUMV_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
SYM_OUMVA_RD_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = RDdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

tm <- Sys.time() - tm
# serialize all the models, WE DO NOT WANT TO SEPND ANOTHER NIGHT REPEATING THIS SHITE
# save(ER_OUM_RD, ER_OUMA_RD, ER_OUMV_RD, ER_OUMVA_RD, ARD_OUM_RD, ARD_OUMA_RD, ARD_OUMV_RD, ARD_OUMVA_RD, SYM_OUM_RD, SYM_OUMA_RD, SYM_OUMV_RD, SYM_OUMVA_RD, file = "../rdata/OU_RD_CD.RData")

save(ER_OUM_RD_CD, ER_OUMA_RD_CD, ER_OUMV_RD_CD, ER_OUMVA_RD_CD, ARD_OUM_RD_CD, ARD_OUMA_RD_CD, ARD_OUMV_RD_CD, ARD_OUMVA_RD_CD, SYM_OUM_RD_CD, SYM_OUMA_RD_CD, SYM_OUMV_RD_CD, SYM_OUMVA_RD_CD, file = "../rdata/OU_RD_CD.RData")
save(ER_OUM_RD_CID, ER_OUMA_RD_CID, ER_OUMV_RD_CID, ER_OUMVA_RD_CID, ARD_OUM_RD_CID, ARD_OUMA_RD_CID, ARD_OUMV_RD_CID, ARD_OUMVA_RD_CID, SYM_OUM_RD_CID, SYM_OUMA_RD_CID, SYM_OUMV_RD_CID, SYM_OUMVA_RD_CID, file = "../rdata/OU_RD_CID.RData")

Warning message in OUwie::hOUwie(phy = COLLAB_395SP_TREE, data = RDdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = COLLAB_395SP_TREE, data = RDdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = COLLAB_395SP_TREE, data = RDdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


...

Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = COLLAB_395SP_TREE, data = RDdata, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = COLLAB_395SP_TREE, data = RDdata, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


In [38]:
# well, damn
tm # Time difference of 15.65342 hours

Time difference of 15.65342 hours

## ___Model fitting with state customizations___
-------------------------------------

In [20]:
# remove ErM records
# change AM/NM to NM


table(MERGED$state)


    AM AM/EcM  AM/NM    EcM    ErM     NM 
   300     15      8     65      3      4 

In [29]:
state_altered <- MERGED[-which(MERGED$state == "ErM"), ] # drop records of ErM
state_altered$state <- gsub(state_altered$state, pattern = "AM/NM", replacement = "NM") # replace AM/NM with NM
state_altered$binominal <- gsub(state_altered$binominal, pattern = ' ', replacement = '_') # replace the spaces in the binominal names with underscores to match the tip labels 

In [30]:
table(state_altered$state) # only got 4 states


    AM AM/EcM    EcM     NM 
   300     15     65     12 

In [33]:
setdiff(PHYLOGENY$tip.label, state_altered$binominal) # ErM species that need to be removed from the phylogeny

[1] "Rhododendron_hypoglaucum" "Vaccinium_mandarinorum"  
[3] "Vaccinium_corymbosum"

In [35]:
# prune off the unneeded (ErM) species from the phylogenetic tree
phylogeny <- ape::drop.tip(PHYLOGENY, tip = setdiff(PHYLOGENY$tip.label, state_altered$binominal), trim.internal = TRUE)
phylogeny


Phylogenetic tree with 392 tips and 391 internal nodes.

Tip labels:
  Anaphalis_aureopunctata, Anaphalis_hancockii, Solidago_decurrens, Doellingeria_scabra, Aster_tataricus, Artemisia_igniaria, ...
Node labels:
  , Spermatophyta, Mesangiospermae, mrcaott2ott121, eudicotyledons, mrcaott2ott969, ...

Rooted; includes branch length(s).

In [39]:
state_altered <- state_altered[match(phylogeny$tip.label, state_altered$binominal), ] # reorder the dataset to match the species order in the phylogeny
stopifnot(all(state_altered$binominal == phylogeny$tip.label))

In [49]:
tm <- Sys.time()

# rate.cat = 1 and null.model = FALSE 

ER_OUM_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ER_OUMA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

SYM_OUM_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE)
SYM_OUMA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = TRUE

ER_OUM_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ER_OUMA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ER_OUMV_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ER_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

ARD_OUM_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ARD_OUMA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ARD_OUMV_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ARD_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

SYM_OUM_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = TRUE)
SYM_OUMA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
SYM_OUMV_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
SYM_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", "state", "F00679")], rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

tm <- Sys.time() - tm
# serialize all the models, WE DO NOT WANT TO SEPND ANOTHER NIGHT REPEATING THIS SHITE

save(ER_OUM_RD_CD, ER_OUMA_RD_CD, ER_OUMV_RD_CD, ER_OUMVA_RD_CD, ARD_OUM_RD_CD, ARD_OUMA_RD_CD, ARD_OUMV_RD_CD, ARD_OUMVA_RD_CD, SYM_OUM_RD_CD, SYM_OUMA_RD_CD, SYM_OUMV_RD_CD, SYM_OUMVA_RD_CD, file = "../rdata/OU_RD_CD_4states.RData")
save(ER_OUM_RD_CID, ER_OUMA_RD_CID, ER_OUMV_RD_CID, ER_OUMVA_RD_CID, ARD_OUM_RD_CID, ARD_OUMA_RD_CID, ARD_OUMV_RD_CID, ARD_OUMVA_RD_CID, SYM_OUM_RD_CID, SYM_OUMA_RD_CID, SYM_OUMV_RD_CID, SYM_OUMVA_RD_CID, file = "../rdata/OU_RD_CID_4states.RData")

Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


...

Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered[, c("binominal", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


## ___RD &le; 1.00 mm cutoff `1005` species with genus level state recommendations from `FungalRoot`___
__________________________

In [3]:
# this data has species averaged root trait (RD, SRL) data for 1005 species where RD is <= 1.00 mm and the root order is 1 or unknown
# and the mycorrhizal state information is the genus level recommendations from the FungalRoot paper supplementary material
collab_1005sp <- read.csv("../../data/chapter2/FREDv3subset/collab_1005sp_states_n_sp_avgd_RD_n_SRL.csv", stringsAsFactors = TRUE)
head(collab_1005sp)

,binominal,state,F00679,F00727,F01286,F01287,F01289,F01290
,<fct>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>
1,Abelia_biflora,AM,0.5292000,30.94706,Abelia,biflora,Caprifoliaceae,Dipsacales
2,Abies_fargesii,EcM,0.2749037,30.65517,Abies,fargesii,Pinaceae,Pinales
3,Abies_nephrolepis,EcM,0.4152333,27.04586,Abies,nephrolepis,Pinaceae,Pinales
4,Acacia_auriculiformis,EcMAM,0.3348000,77.29000,Acacia,auriculiformis,Fabaceae,Fabales
5,Acacia_crassicarpa,EcMAM,0.2259300,108.28000,Acacia,crassicarpa,Fabaceae,Fabales
6,Acacia_mangium,EcMAM,0.2525500,99.19000,Acacia,mangium,Fabaceae,Fabales


In [5]:
phylo1005sp <- ape::multi2di(ape::read.tree(file = "../../data/chapter2/uphylomaker/FRED_subset_collab_1005sp.tre"))

RD1005sp <- collab_1005sp[, c("binominal", "state", "F00679")]
RD1005sp$binominal <- RD1005sp$binominal[match(phylo1005sp$tip.label, RD1005sp$binominal)]
stopifnot(all(RD1005sp$binominal == phylo1005sp$tip.label))

In [6]:
table(RD1005sp$state) # 7 states (1 Orchidaceous)


   AM   EcM EcMAM   ErM    NM  NMAM    OM 
  777   122    26    10    20    49     1 

In [ ]:
# this will probably take days

tm <- Sys.time()

# rate.cat = 1 and null.model = FALSE 

ER_OUM_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ER_OUMA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

SYM_OUM_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE)
SYM_OUMA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_RD_CD <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = TRUE

ER_OUM_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ER_OUMA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ER_OUMV_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ER_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

ARD_OUM_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ARD_OUMA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ARD_OUMV_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ARD_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

SYM_OUM_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = TRUE)
SYM_OUMA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
SYM_OUMV_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
SYM_OUMVA_RD_CID <- OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

tm <- Sys.time() - tm

save(ER_OUM_RD_CD, ER_OUMA_RD_CD, ER_OUMV_RD_CD, ER_OUMVA_RD_CD, ARD_OUM_RD_CD, ARD_OUMA_RD_CD, ARD_OUMV_RD_CD, ARD_OUMVA_RD_CD, SYM_OUM_RD_CD, SYM_OUMA_RD_CD, SYM_OUMV_RD_CD, SYM_OUMVA_RD_CD, file = "../rdata/OU_RD_CD_1005sp.RData")
save(ER_OUM_RD_CID, ER_OUMA_RD_CID, ER_OUMV_RD_CID, ER_OUMVA_RD_CID, ARD_OUM_RD_CID, ARD_OUMA_RD_CID, ARD_OUMV_RD_CID, ARD_OUMVA_RD_CID, SYM_OUM_RD_CID, SYM_OUMA_RD_CID, SYM_OUMV_RD_CID, SYM_OUMVA_RD_CID, file = "../rdata/OU_RD_CID_1005sp.RData")

Warning message in OUwie::hOUwie(phy = phylo1005sp, data = RD1005sp, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


## ___Dual Transitions (mycorrhizal states & photosynthetic pathways)___
-----------------